In [1]:
# Path: MobileNet.ipynb
# Abnormal Crowd Behavior Detection Using MobileNet
# Labels: ['filename', ' Diff_Direction', ' Non_Pedestrain', ' Opp_Direction', ' Running', ' Sitting', ' Sleeping', ' Standing'] 7 labels
# Each imgae is 400x400x3
# dataset is aready split into train, test and valid each folder contains an csv file (_classes.csv) with each image labels
# 7939 train
# 378 test
# 757 valid
# 9074 total
# For a multi-label problem, where each sample can have multiple labels rather than just a single label, you will need to modify the final layers of the MobileNet model:

# 1- The global average pooling layer will stay the same.

# 2- Instead of having a single node output layer with softmax activation for classification, you would have multiple nodes equal to the number of total possible labels.

# 3- Use a sigmoid activation function rather than softmax, since this is now a multi-label binary classification problem rather than single label classification.

# * During training, use binary cross-entropy loss rather than categorical cross-entropy, since now each label is treated independently.

# * Make sure your target labels are encoded as 0/1 rather than one-hot to match the sigmoid output and binary cross-entropy loss. (Done)

# * All other aspects of training MobileNet like hyperparameters, data augmentation etc. can remain the same for a multi-label task. (Noted)

# Libraries
import cv2
import pandas as pd
import numpy as np
import itertools
import random

import os
import shutil

from tqdm import tqdm
import sys
import glob

# Model Libraries
import tensorflow as tf
from tensorflow.keras.layers import Dense, Activation, GlobalAveragePooling2D, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import categorical_crossentropy
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras.models import Model
from tensorflow.keras.applications.imagenet_utils import decode_predictions

from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt
%matplotlib inline
# End Model Libraries

# print(tf.config.list_physical_devices('GPU'))

def beep(frequency = 2500, duration = 1000):
    if os.name == 'nt':
        from winsound import Beep
        Beep(frequency, duration)
    else:
        os.system('play --no-show-progress --null --channels 1 synth %s sine %f' % (duration / 1000, frequency))

def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

# Dirs
train_path = "../dataset/train"
test_path = "../dataset/test"
valid_path = "../dataset/valid"

train_updatednames_path = "../dataset/output/train_updatednames"
test_updatednames_path = "../dataset/output/test_updatednames"
valid_updatednames_path = "../dataset/output/valid_updatednames"

train_splited_path = "../dataset/output/train_splited"
test_splited_path = "../dataset/output/test_splited"
valid_splited_path = "../dataset/output/valid_splited"

CSV_FILENAME = "_classes.csv"

LABELS = [' Diff_Direction',' Non_Pedestrain',' Opp_Direction',' Running',' Sitting',' Sleeping',' Standing']


In [ ]:
def copy_images_with_updated_filenames(input_path, output_path):

    create_folder(output_path)

    csv_path = os.path.join(input_path, CSV_FILENAME)

    df = pd.read_csv(csv_path)

    total = len(df)
    pbar = tqdm(total=total)

    # Iterate over each row in the DataFrame
    for _, row in df.iterrows():
        image_filename = row['filename']
        image_classes = row[LABELS] # Get the labels for this image

        image_path = os.path.join(input_path, image_filename)

        selected_labels = [label for label, value in image_classes.items() if value == 1]

        selected_labels = [label.replace(" ", "") for label in selected_labels]

        updated_filename = f"{os.path.splitext(image_filename)[0]}-{'-'.join(selected_labels)}.jpg"

        new_image_path = os.path.join(output_path, updated_filename)      

        try:
            shutil.copyfile(image_path, new_image_path)
        except Exception as e:
            print(new_image_path)
            print("#"*50)
            print(e)
            print("#"*50)

        pbar.update(1)

    pbar.close()
    # count number of images
    image_number = len(glob.glob(output_path + "/*.jpg"))
    print(f"{image_number} images in {output_path}")

In [ ]:
copy_images_with_updated_filenames(train_path, train_updatednames_path)
copy_images_with_updated_filenames(valid_path, valid_updatednames_path)
copy_images_with_updated_filenames(test_path, test_updatednames_path)

In [ ]:
def split_images(input_path, output_path):

    create_folder(output_path)

    total = len(os.listdir(input_path))
    pbar = tqdm(total=total)

    labels = [label.strip() for label in LABELS]
    for label in labels:
        folder_path = os.path.join(output_path, label)
        create_folder(folder_path)

    for filename in os.listdir(input_path):
        for label in labels:
            if label in filename:
                src = os.path.join(input_path, filename)
                dest = os.path.join(output_path, label, filename)
                try:
                    shutil.copyfile(src, dest)
                except Exception as e:
                    print(dest)
                    print("#"*50)
                    print(e)
                    print("#"*50)
        pbar.update(1)

    pbar.close()

In [ ]:
split_images(train_updatednames_path, train_splited_path)
split_images(valid_updatednames_path, valid_splited_path)
split_images(test_updatednames_path, test_splited_path)

In [ ]:
def count_images_in_each_label_folder(path):
    labels = [label.strip() for label in LABELS]
    for label in labels:
        folder_path = os.path.join(path, label)
        image_number = len(glob.glob(folder_path + "/*.jpg"))
        print(f"{image_number} images in {folder_path}")

In [ ]:
count_images_in_each_label_folder(train_splited_path)
print("#"*50)
count_images_in_each_label_folder(valid_splited_path)
print("#"*50)
count_images_in_each_label_folder(test_splited_path)

In [2]:
# Define the batch size
batch_size = 32

# Define the image size
image_size = (224, 224)

# Define the train, test and validation data generators
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)
valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_splited_path,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical')

test_generator = test_datagen.flow_from_directory(
    test_splited_path,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical')

valid_generator = valid_datagen.flow_from_directory(
    valid_splited_path,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical')


Found 31401 images belonging to 7 classes.
Found 1480 images belonging to 7 classes.
Found 2937 images belonging to 7 classes.


In [3]:
assert train_generator.n == 31401
assert test_generator.n == 1480
assert valid_generator.n == 2937
assert train_generator.num_classes == valid_generator.num_classes == test_generator.num_classes == 7

In [4]:
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.layers import Dense

# Load the MobileNet model
base_model = MobileNet(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Add new layers to the model
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(7, activation='sigmoid')(x)

# Define the new model
model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [5]:
model.fit(
    train_generator,
    epochs=5,
    validation_data=valid_generator,
    verbose=1)

Epoch 1/5
982/982 [==============================] - 3680s 4s/step - loss: 0.3689 - accuracy: 0.2310 - val_loss: 0.3516 - val_accuracy: 0.2428
Epoch 2/5
411/982 [===========>..................] - ETA: 34:32 - loss: 0.3541 - accuracy: 0.2284